In [1]:
import pandas as pd
data= pd.read_csv("rxdx_aug_2026_indexed_data_med_mod.csv")
data.head()

,description,NDC,HCPCS
0,"""HYDROCODONE-ACETAMINOPHEN 7.5-325 MG/15 ML SO...",53217-0136-01,NaN
1,"""HYDROCODONE-ACETAMINOPHEN 7.5-325 MG/15 ML SO...",53217-0136-01,NaN
2,(DAUNORUBICIN AND CYTARABINE) LIPOSOME 100 MG/...,68727-0745-02,J9100
3,(DAUNORUBICIN AND CYTARABINE) LIPOSOME 100 MG/...,68727-0745-02,NaN
4,"(TO DELIVER) AVOBENZONE 2%, HOMOSALATE 8%, OCT...",66800-3351-07,NaN


In [2]:
# doc=[]
# for index, row in data.iterrows():
#     text = f"""
#     NDC: {row['NDC']}
#     HCPCS: {row['HCPCS']}
#     Description: {row['description']}
#     """
#     doc.append(text)
data["text"] = data["description"].fillna("") + " ," + data["NDC"].fillna("") + ", " + data["HCPCS"].fillna("")

In [3]:
data["text"][0]

'"HYDROCODONE-ACETAMINOPHEN 7.5-325 MG/15 ML SOLN 10 ML CUP" ,53217-0136-01, '

In [21]:
from sentence_transformers import SentenceTransformer
import faiss

In [22]:
model = SentenceTransformer('pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7882.17it/s]


In [23]:
from tqdm import tqdm

def compute_embeddings(descriptions):
    all_embeddings = []
    CHUNK_SIZE = 1000
    for i in tqdm(range(0, len(descriptions), CHUNK_SIZE)):
        chunk = descriptions[i:i + CHUNK_SIZE]
        embeddings = model.encode(
            chunk,
            batch_size=128,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        all_embeddings.extend(
            embeddings.tolist()
        )

    return all_embeddings

In [4]:
texts = data["text"].astype(str).to_list()

In [27]:
embeddings = compute_embeddings(texts)

100%|██████████| 443/443 [6:52:48<00:00, 55.91s/it]  


In [28]:
data["embeddings"]= embeddings

In [29]:
data.head()

,description,NDC,HCPCS,text,embeddings
0,"""HYDROCODONE-ACETAMINOPHEN 7.5-325 MG/15 ML SO...",53217-0136-01,NaN,"""HYDROCODONE-ACETAMINOPHEN 7.5-325 MG/15 ML SO...","[-0.014046110212802887, -0.03199256956577301, ..."
1,"""HYDROCODONE-ACETAMINOPHEN 7.5-325 MG/15 ML SO...",53217-0136-01,NaN,"""HYDROCODONE-ACETAMINOPHEN 7.5-325 MG/15 ML SO...","[-0.025276441127061844, -0.01146012358367443, ..."
2,(DAUNORUBICIN AND CYTARABINE) LIPOSOME 100 MG/...,68727-0745-02,J9100,(DAUNORUBICIN AND CYTARABINE) LIPOSOME 100 MG/...,"[-0.02556862309575081, 0.05774054676294327, 0...."
3,(DAUNORUBICIN AND CYTARABINE) LIPOSOME 100 MG/...,68727-0745-02,NaN,(DAUNORUBICIN AND CYTARABINE) LIPOSOME 100 MG/...,"[-0.022312214598059654, 0.04458283632993698, -..."
4,"(TO DELIVER) AVOBENZONE 2%, HOMOSALATE 8%, OCT...",66800-3351-07,NaN,"(TO DELIVER) AVOBENZONE 2%, HOMOSALATE 8%, OCT...","[-0.060332149267196655, -0.003652493003755808,..."


In [30]:
data.to_csv("rxdx_aug_2026_indexed_data_med_mod_embeddings.csv", index=False)

In [32]:
import json

data["embeddings"] = data["embeddings"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

In [35]:
import numpy as np
fda_embeddings = np.vstack(data["embeddings"].values).astype("float32")

In [37]:
dimension = fda_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(fda_embeddings)

In [39]:
faiss.write_index(index,"rxdx_aug_2026_index_med_mod_updated.faiss")

BM25 Keyword Search/Elastic search

In [ ]:
from elasticsearch import Elasticsearch
from dotenv import load_dotenv
load_dotenv()
import os
password= os.getenv('Elastic_pass')
username= os.getenv("username")
es = Elasticsearch(
    "https://localhost:9200",
    basic_auth=(username, password),
    verify_certs=False
)

# print(es.info())

C:\Users\pthorat\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\elasticsearch\_sync\client\__init__.py:326: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(
C:\Users\pthorat\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'name': '2020AI-050', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'nz0GDvzFQ_i29N0F_Eba1Q', 'version': {'number': '9.4.1', 'build_flavor': 'default', 'build_type': 'zip', 'build_hash': '3c7c6027c5769d860d87448e2749f4c550a239da', 'build_date': '2026-05-08T10:08:29.383338563Z', 'build_snapshot': False, 'lucene_version': '10.4.0', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


In [7]:
mapping = {
    "mappings": {
        "properties": {

            "description": {
                "type": "text"
            },

            "NDC": {
                "type": "keyword"
            },

            "HCPCS": {
                "type": "keyword"
            }
        }
    }
}

In [8]:
INDEX_NAME="rxdx_aug2026_index"

In [9]:
if not es.indices.exists(index=INDEX_NAME):

    es.indices.create(
        index=INDEX_NAME,
        body=mapping
    )

print("Index Created")

Index Created


C:\Users\pthorat\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [10]:
data = data.fillna("")

In [12]:
from tqdm import tqdm
from elasticsearch.helpers import bulk
def generate_documents(data):

    for idx, row in tqdm(data.iterrows(), total=len(data)):

        yield {
            "_index": INDEX_NAME,

            "_id": idx,

            "_source": {

                "description": str(row["description"]),

                "HCPCS": str(row["HCPCS"]),

                "NDC": str(row["NDC"])
            }
        }

# Bulk insert
bulk(
    es,
    generate_documents(data)
)

print("Indexing Completed")

  0%|          | 0/442211 [00:00<?, ?it/s]

C:\Users\pthorat\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
  0%|          | 501/442211 [00:00<04:45, 1549.19it/s]C:\Users\pthorat\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\pthorat\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-pa

Indexing Completed



C:\Users\pthorat\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [13]:
print(
    es.count(index=INDEX_NAME)
)

C:\Users\pthorat\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'count': 442211, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}}


In [14]:
print(
    es.indices.get_alias().keys()
)

dict_keys(['.security-7', 'rxdx_aug2026_index'])


C:\Users\pthorat\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\pthorat\AppData\Local\Temp\ipykernel_3772\981748270.py:2: ElasticsearchWarning: this request accesses system indices: [.security-7], but in a future major version, direct access to system indices will be prevented by default
  es.indices.get_alias().keys()
